<a href="https://colab.research.google.com/github/k98j/Research-Projects/blob/main/AlexNet_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn


class CustomLRN(nn.Module):
    """Local Response Normalization (LRN) matching AlexNet defaults."""
    def __init__(self, size=5, alpha=1e-4, beta=0.75, k=2.0):
        super(CustomLRN, self).__init__()
        self.lrn = nn.LocalResponseNorm(size=size, alpha=alpha, beta=beta, k=k)

    def forward(self, x):
        return self.lrn(x)


class AlexNet(nn.Module):
    def __init__(self, num_classes=1000):
        super(AlexNet, self).__init__()

        # Feature extraction layers: Convolutional, Activation, Normalization, and Pooling layers
        self.feature_extractor = nn.Sequential(
            # Conv1: 11x11, Stride 4, Padding 2 -> Output: (96, 55, 55)
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            CustomLRN(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),  # Output: (96, 27, 27)

            # Conv2: 5x5, Padding 2 -> Output: (256, 27, 27)
            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            CustomLRN(size=5, alpha=1e-4, beta=0.75, k=2.0),
            nn.MaxPool2d(kernel_size=3, stride=2),  # Output: (256, 13, 13)

            # Conv3: 3x3, Padding 1 -> Output: (384, 13, 13)
            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv4: 3x3, Padding 1 -> Output: (384, 13, 13)
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv5: 3x3, Padding 1 -> Output: (256, 13, 13)
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),  # Output: (256, 6, 6)
        )

        # Adaptive pooling layer ensures spatial shape (6, 6) regardless of slight input size variations
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(6, 6))

        # Fully connected layers for classification
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.feature_extractor(x)  # Pass through convolutional feature extractor
        x = self.avgpool(x)            # Ensure (256, 6, 6) output dimension
        x = torch.flatten(x, 1)        # Flatten tensor for fully connected layers (batch_size, 256*6*6)
        x = self.classifier(x)         # Pass through fully connected classifier
        return x


# Quick dimensional test run
if __name__ == "__main__":
    model = AlexNet(num_classes=1000)
    dummy_input = torch.randn(2, 3, 224, 224)  # Standard input batch size of 2
    output = model(dummy_input)
    print(f"Input shape:  {dummy_input.shape}")
    print(f"Output shape: {output.shape}")

Input shape:  torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 1000])
